In [5]:
import os
os.path.join("out_dir", "AIS_samples.jsonl")

'out_dir/AIS_samples.jsonl'

In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from rashomon import hasse, extract_pools, loss, aggregate, AIS, MCMC

statistic = "mean"

# N_ITER = 30000
# N_BURN = 10000
# N_THIN = 10
# n_paths = 300
# n_levels = 20

N_ITER = 110
N_BURN = 10
N_THIN = 1
n_paths=30
n_levels=5

moves_per_level=5
num_samples_per_feature=500

lamb=1

M = 2
R = np.array([4, 3])

num_profiles = 2**M
profiles, profile_map = hasse.enumerate_profiles(M)

all_policies = hasse.enumerate_policies(M, R)
num_policies = len(all_policies)

sigma_00 = None
mu_00 = np.array([0])
# mu_00 = np.array([5])
var_00 = np.array([1])

# Profile (0, 1)
sigma_01 = np.array([[1]])
mu_01 = np.array([-1])
# mu_01 = np.array([10])
var_01 = np.array([1])

# Profile (1, 0)
sigma_10 = np.array([[1, 0]])
mu_10 = np.array([-2, -3])
# mu_10 = np.array([-10, 15])
var_10 = np.array([1, 1])

# Profile (1, 1)
sigma_11 = np.array([[0, 1], [0, np.inf]])
mu_11 = np.array([2, 3, -1, 1])
# mu_11 = np.array([20, 30, 5, 10])
var_11 = np.array([1, 1, 1, 1])

sigma = [sigma_00, sigma_01, sigma_10, sigma_11]
mu = [mu_00, mu_01, mu_10, mu_11]
var = [var_00, var_01, var_10, var_11]

policies_profiles = {}
policies_profiles_masked = {}
policies_ids_profiles = {}
pi_policies = {}
pi_pools = {}
for k, profile in enumerate(profiles):

    policies_temp = [(i, x) for i, x in enumerate(all_policies) if hasse.policy_to_profile(x) == profile]
    unzipped_temp = list(zip(*policies_temp))
    policies_ids_k = list(unzipped_temp[0])
    policies_k = list(unzipped_temp[1])
    policies_profiles[k] = deepcopy(policies_k)
    policies_ids_profiles[k] = policies_ids_k

    profile_mask = list(map(bool, profile))

    # Mask the empty arms
    for idx, pol in enumerate(policies_k):
        policies_k[idx] = tuple([pol[i] for i in range(M) if profile_mask[i]])
    policies_profiles_masked[k] = policies_k

    if np.sum(profile) > 0:
        pi_pools_k, pi_policies_k = extract_pools.extract_pools(policies_k, sigma[k])
        if len(pi_pools_k.keys()) != mu[k].shape[0]:
            print(f"Profile {k}. Expected {len(pi_pools_k.keys())} pools. Received {mu[k].shape[0]} means.")
        pi_policies[k] = pi_policies_k
        # pi_pools_k has indicies that match with policies_profiles[k]
        # Need to map those indices back to all_policies
        pi_pools[k] = {}
        for x, y in pi_pools_k.items():
            y_full = [policies_profiles[k][i] for i in y]
            y_agg = [all_policies.index(i) for i in y_full]
            pi_pools[k][x] = y_agg
    else:
        pi_policies[k] = {0: 0}
        pi_pools[k] = {0: [0]}

def generate_data(mu, var, n_per_pol, all_policies, pi_policies, M):
    num_data = num_policies * n_per_pol
    X = np.zeros(shape=(num_data, M))
    D = np.zeros(shape=(num_data, 1), dtype='int_')
    y = np.zeros(shape=(num_data, 1))

    idx_ctr = 0
    for k, profile in enumerate(profiles):
        policies_k = policies_profiles[k]

        for idx, policy in enumerate(policies_k):
            policy_idx = [i for i, x in enumerate(all_policies) if x == policy]

            pool_id = pi_policies[k][idx]
            mu_i = mu[k][pool_id]
            var_i = var[k][pool_id]
            y_i = np.random.normal(mu_i, var_i, size=(n_per_pol, 1))

            start_idx = idx_ctr * n_per_pol
            end_idx = (idx_ctr + 1) * n_per_pol

            X[start_idx:end_idx, ] = policy
            D[start_idx:end_idx, ] = policy_idx[0]
            y[start_idx:end_idx, ] = y_i

            idx_ctr += 1

    return X, D, y

num_samples_per_feature = num_samples_per_feature

overall_result = []

for iter in range(1):
    np.random.seed(iter*42)
    X, D, y = generate_data(mu, var, num_samples_per_feature, all_policies, pi_policies, M)
    policy_means = loss.compute_policy_means(D, y, num_policies)
    prof_idx_of_policy, profiles = AIS.build_profile_index_of_policy(all_policies, hasse.policy_to_profile)


    def score_s(state):
        Q = AIS.global_loss_raw(
            state=state,
            D=D, y=y, M=M, R=R,
            prof_idx_of_policy = prof_idx_of_policy,
            policies=all_policies,
            policy_means=policy_means,
            reg=lamb, normalize=0,
            lattice_edges=None,
        )
        return float(np.exp(-Q))

    result = dict()

    if statistic == "mean":
        ### MCMC
        start = time.time()
        MCMC.run_mcmc_streaming_rand_start(
            profiles = profiles,
            M = M,
            R = R,
            score_s = score_s,
            seed = None,
            steps = N_ITER,
            burnin = N_BURN,
            thin = N_THIN,
            min_len = 1,
            out_jsonl = "./output_files/mcmc_samples.jsonl",
            progress_json = "./output_files/mcmc_progress.json"
        )

        mcmc_res = MCMC.load_mcmc_res_from_jsonl("./output_files/mcmc_samples.jsonl")

        MCMC_post_mean = MCMC.policy_means_matrix_from_mcmc(
            mcmc_res["samples"],                      # mcmc_res["samples"]
            all_policies,                     # global list/array of policies (length P)
            policy_means,                 # np.ndarray [P,2] = [sum_y, count]
            prof_idx_of_policy,           # length-P array: policy_id -> profile k
            R,                        # np.ndarray of arm levels (includes control)
            M,
            lattice_edges=None,           # pass None if unused
            policy_labels=None            # optional column names; defaults to policy indices (0..P-1)
        )

        end = time.time()

        result["MCMC_time"] = end - start
        result["MCMC"] = np.mean(MCMC_post_mean, axis=0)

        ### Exact:
        start = time.time()

        all_partitions, losses = AIS.enumerate_all_states_and_losses(
            profiles=profiles,
            R=R,
            M=M,
            policies=all_policies,
            policy_means=policy_means,
            prof_idx_of_policy=prof_idx_of_policy,
            D=D, y=y,
            reg=lamb, normalize=0,
            lattice_edges=None,
            max_states=None  # or an integer cap to safeguard
        )

        true_log_post = [-i[1] for i in losses]

        exact_post_mean = AIS.estimate_policy_means_from_RPS(
            all_partitions,                     # dict with key "samples": List[State]
            true_log_post,
            all_policies,                     # global policy list (length P)
            policy_means,                 # np.ndarray [P,2] = [sum_y, count]
            prof_idx_of_policy,           # length-P array: policy_id -> profile k, for 36 policies, which profile is each policy in
            R,                        # np.ndarray of arm levels (includes control)
            M,
            lattice_edges=None            # optional lattice; pass None if unused
        )

        end = time.time()

        result["exact_time"] = end-start
        result["exact"] = exact_post_mean

        ### AIS / RPS

        for theta in [7.8, 8, 8.2, 8.4, 8.6]:
            start = time.time()
            H = np.inf
            R_set, R_profiles = aggregate.RAggregate(M, R, H, D, y, theta, reg=lamb, verbose=True)

            anchors = AIS.build_anchor_states(R_set, R_profiles, M, R)
            prof_idx_of_policy, profiles = AIS.build_profile_index_of_policy(all_policies, hasse.policy_to_profile)

            RPS_states = AIS.raggregate_to_states((R_set, R_profiles), profiles)

            log_alpha = [np.log(max(1e-300, score_s(A))) for A in anchors]

            RPS_post_mean = AIS.estimate_policy_means_from_RPS(
                RPS_states,                     # dict with key "samples": List[State]
                log_alpha,
                all_policies,                     # global policy list (length P)
                policy_means,                 # np.ndarray [P,2] = [sum_y, count]
                prof_idx_of_policy,           # length-P array: policy_id -> profile k, for 36 policies, which profile is each policy in
                R,                        # np.ndarray of arm levels (includes control)
                M,
                lattice_edges=None            # optional lattice; pass None if unused
            )

            end = time.time()
            
            result[f"RPS_{theta}_time"] = end-start
            result[f"RPS_{theta}"] = RPS_post_mean


            start = time.time()
            cfg = AIS.AISConfig(n_paths=n_paths, n_levels=n_levels, moves_per_level=moves_per_level, min_len=1, seed=2)
            ais_out = AIS.run_ais_streaming_from_data(
                M, 
                R, 
                H, 
                D, 
                y, 
                theta, 
                reg=lamb,
                eps1=0.5,
                eps2=0.75,
                all_policies=all_policies,
                score_s=score_s,
                out_dir="./output_files",
                cfg = cfg,
            )

            mu_hat = AIS.estimate_policy_means_from_ais(
                ais_out=ais_out,                 # from your run_ais_state
                all_policies=all_policies,
                policy_means=policy_means,
                prof_idx_of_policy=prof_idx_of_policy,
                lattice_edges=None,                  # or the hasse edges if you use them
                R_per = R,
                M=M
            )

            end = time.time()

            result[f"AIS_{theta}_time"] = end - start
            result[f"AIS_{theta}"] = mu_hat

    overall_result.append(result)

with open("./output/sim1_result.pkl", "wb") as f:
    pickle.dump(result, f)

(0, 0) 6.913983213235163
Profile (0, 0) has 1 objects in Rashomon set
(0, 1) 6.987866083122113
Adaptive
Profile (0, 1) took 0.0015110969543457031 s adaptively
Profile (0, 1) has 2 objects in Rashomon set
(1, 0) 7.061153034821505
Adaptive
Profile (1, 0) took 0.003084897994995117 s adaptively
Profile (1, 0) has 4 objects in Rashomon set
(1, 1) 7.329868070487803
Adaptive
Profile (1, 1) took 0.012598037719726562 s adaptively
Profile (1, 1) has 8 objects in Rashomon set
Finding feasible combinations
Min = 6.104129726871667, Max = 12.96904319944447
(0, 0) 6.913983213235163
Profile (0, 0) has 1 objects in Rashomon set
(0, 1) 6.987866083122113
Adaptive
Profile (0, 1) took 0.0014810562133789062 s adaptively
Profile (0, 1) has 2 objects in Rashomon set
(1, 0) 7.061153034821505
Adaptive
Profile (1, 0) took 0.0030951499938964844 s adaptively
Profile (1, 0) has 4 objects in Rashomon set
(1, 1) 7.329868070487803
Adaptive
Profile (1, 1) took 0.013541221618652344 s adaptively
Profile (1, 1) has 8 obje

In [13]:
with open("./output/sim1_result.pkl", "wb") as f:
    pickle.dump([result], f)

with open("./output/sim1_result.pkl", 'rb') as f:
    x = pickle.load(f)


In [64]:
with open("./output/sim1_result.pkl", 'rb') as f:
    x = pickle.load(f)

with open("./output/sim1_result2.pkl", 'rb') as f:
    x2 = pickle.load(f)

In [56]:
x

[{'MCMC_time': 1.098031997680664,
  'MCMC':           0         1         2         3         4         5         6   \
  0  -0.025354 -1.017981 -1.017981 -2.368140  0.830062  0.830062 -2.368140   
  1  -0.025354 -1.017981 -1.017981 -2.368140  0.830062  0.830062 -2.368140   
  2  -0.025354 -1.017981 -1.017981 -2.368140  0.830062  0.830062 -2.368140   
  3  -0.025354 -1.017981 -1.017981 -2.368140  0.830062  0.830062 -2.368140   
  4  -0.025354 -1.017981 -1.017981 -2.368140  0.830062  0.830062 -2.368140   
  ..       ...       ...       ...       ...       ...       ...       ...   
  95 -0.025354 -1.017981 -1.017981 -2.015678  0.830062  0.830062 -2.015678   
  96 -0.025354 -1.017981 -1.017981 -2.015678  0.830062  0.830062 -2.015678   
  97 -0.025354 -1.017981 -1.017981 -2.015678  0.830062  0.830062 -2.015678   
  98 -0.025354 -1.017981 -1.017981 -2.015678  2.481057  2.481057 -2.015678   
  99 -0.025354 -1.017981 -1.017981 -2.015678  2.481057  2.481057 -2.015678   
  
            7      

In [65]:
x2

{'RPS_7.8_time': 2.2231884002685547,
 'RPS_7.8': array([-0.02535444, -1.02507505, -1.01088692, -2.25579441,  1.23371018,
         1.58199611, -2.33975121,  0.41897682,  0.76726276, -2.50887463,
         0.31506892,  0.66335486]),
 'AIS_7.8_time': 55.991974115371704,
 'AIS_7.8': array([-0.02535444, -1.03236704, -1.00359492, -2.22734842,  1.1415602 ,
         1.64917732, -2.31262134,  0.38640018,  0.89401729, -2.5644505 ,
         0.19303729,  0.71617736]),
 'RPS_8_time': 2.20864200592041,
 'RPS_8': array([-0.02535444, -1.02575701, -1.01020495, -2.24499444,  1.23544435,
         1.55484355, -2.33702215,  0.48828477,  0.80768396, -2.52240367,
         0.28735692,  0.60675611]),
 'AIS_8_time': 55.78394389152527,
 'AIS_8': array([-0.02535444, -1.03199675, -1.00396522, -2.23419339,  1.56443313,
         1.99143871, -2.33848744,  0.18465678,  0.71634058, -2.53173943,
        -0.00409167,  0.52759213]),
 'RPS_8.2_time': 2.211799144744873,
 'RPS_8.2': array([-0.02535444, -1.02709946, -1.0088625

In [62]:
import importlib
importlib.reload(AIS)
cfg = AIS.AISConfig(n_paths=n_paths, n_levels=n_levels, moves_per_level=moves_per_level, min_len=1, seed=2)

ais_out = AIS.run_ais_streaming_from_data_parallel(
                M, 
                R, 
                H, 
                x,
                D, 
                y, 
                theta, 
                reg=lamb,
                eps1=0.5,
                eps2=0.75,
                all_policies=all_policies,
                policy_means=policy_means,
                score_s=score_s,
                out_dir="./output_files",
                num_workers=6,
                cfg = cfg,
            )

(0, 0) 7.713983213235162
Profile (0, 0) has 1 objects in Rashomon set
(0, 1) 7.787866083122113
Adaptive
Profile (0, 1) took 0.0014638900756835938 s adaptively
Profile (0, 1) has 2 objects in Rashomon set
(1, 0) 7.861153034821504
Adaptive
Profile (1, 0) took 0.0030660629272460938 s adaptively
Profile (1, 0) has 4 objects in Rashomon set
(1, 1) 8.129868070487802
Adaptive
Profile (1, 1) took 0.012999773025512695 s adaptively
Profile (1, 1) has 8 objects in Rashomon set
Finding feasible combinations
Min = 6.104129726871667, Max = 12.96904319944447
Running pilot adaptive ladder
Adaptive ladder has 6 levels; first 10: [0.   0.05 0.15 0.35 0.75 1.  ]
Per-step ESS ratios (len = 5 ): [1.0, 0.9913673317344911, 0.9661198469626824, 0.9435768657754946, 0.9865325923648947]


In [63]:
ais_out = AIS.run_ais_streaming_from_data(
                M, 
                R, 
                H, 
                D, 
                y, 
                theta, 
                reg=lamb,
                eps1=0.5,
                eps2=0.75,
                all_policies=all_policies,
                score_s=score_s,
                out_dir="./output_files",
                cfg = cfg,
            )

(0, 0) 7.713983213235162
Profile (0, 0) has 1 objects in Rashomon set
(0, 1) 7.787866083122113
Adaptive
Profile (0, 1) took 0.002752065658569336 s adaptively
Profile (0, 1) has 2 objects in Rashomon set
(1, 0) 7.861153034821504
Adaptive
Profile (1, 0) took 0.003144979476928711 s adaptively
Profile (1, 0) has 4 objects in Rashomon set
(1, 1) 8.129868070487802
Adaptive
Profile (1, 1) took 0.013110160827636719 s adaptively
Profile (1, 1) has 8 objects in Rashomon set
Finding feasible combinations
Min = 6.104129726871667, Max = 12.96904319944447
Running pilot adaptive ladder
Adaptive ladder has 6 levels; first 10: [0.   0.05 0.15 0.35 0.75 1.  ]
Per-step ESS ratios (len = 5 ): [1.0, 0.9913673317344911, 0.9661198469626824, 0.9435768657754946, 0.9865325923648947]


In [59]:
len(ais_out.terminals)

30